# Houseplant SFT v2 — Colab T4

Обучение на `data/sft_v2/sft_train.jsonl` (1 060 пар, отбор судьёй, см. `reports/sft_v2_data.md`) → адаптер `artifacts/weights/s2_sft_v2` → копия на Drive `MyDrive/rk1_houseplant/s2_sft_v2`. Оценка — `notebooks/rk1_eval_overnight.ipynb`.

Стек (pinned): `transformers==4.57.1` + `trl==0.24.0` + `peft==0.17.1` + `bitsandbytes` (Linux).

Перед запуском:
1. Runtime → GPU (T4).
2. Если после старого `--force-reinstall` сломан numpy/torch: **Runtime → Disconnect and delete runtime**, затем новый сеанс.
3. Склонируйте репозиторий (ячейка ниже).
4. Не коммитьте `OPENROUTER_API_KEY`; для judge — Colab Secrets.
5. Не используйте `pip install -U` / `--force-reinstall` на весь файл требований.

In [ ]:
!git clone https://github.com/Schaft-s/LLM || (cd LLM && git pull)
REPO = '/content/LLM'
import os, sys
os.chdir(REPO)
sys.path.insert(0, REPO)
!nvidia-smi || true
!wc -l data/sft_v2/sft_train.jsonl  # expect 1060

In [ ]:
# Safe install: only our pins. NEVER --force-reinstall the whole Colab env.
# Needs bitsandbytes>=0.49.2 for Colab CUDA 13 (older 0.45 = CPU-only → QLoRA crash).
# If numpy/_slice ImportError already happened: Disconnect and delete runtime first.
!bash scripts/colab_install.sh
!python scripts/smoke_colab_stack.py
!python -c "import torch,bitsandbytes as b; from pathlib import Path; print('cuda', torch.cuda.is_available(), torch.version.cuda); print([p.name for p in Path(b.__file__).parent.glob('libbitsandbytes_cuda*.so')])"

## Pilot (20 steps): train → save adapter

In [ ]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, 'scripts/train_qlora.py', '--pilot'],
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise SystemExit(result.returncode)
print('Pilot OK → artifacts/weights/pilot_qlora')

## Reload smoke + 1 generation

In [ ]:
from src.train.generate import generate_chat, load_base_or_adapter
from src.train.sft_qlora import load_yaml

model_cfg = load_yaml('configs/model.yaml')
model, tok = load_base_or_adapter(model_cfg, adapter_dir='artifacts/weights/pilot_qlora')
prompt = (
    'My snake plant has mushy leaves at the base after weekly watering in low light. '
    'What should I check first?'
)
text = generate_chat(
    model, tok, prompt,
    system_prompt=model_cfg.get('system_prompt', '').strip(),
    max_new_tokens=256,
    temperature=0.2,
)
print(text)

## S2 full QLoRA on SFT v2 (after pilot OK)

In [ ]:
result = subprocess.run([sys.executable, 'scripts/train_qlora.py'], capture_output=True, text=True)
print(result.stdout[-5000:])
if result.returncode: print(result.stderr[-5000:]); raise SystemExit(result.returncode)
print('S2 OK → artifacts/weights/s2_sft_v2')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/rk1_houseplant
!cp -r /content/LLM/artifacts/weights/s2_sft_v2 /content/drive/MyDrive/rk1_houseplant/
!ls /content/drive/MyDrive/rk1_houseplant/s2_sft_v2
drive.flush_and_unmount()

## Eval helpers

Полный eval (generate → score → judge → выбор чекпоинта → test → графики) — `notebooks/rk1_eval_overnight.ipynb` / `scripts/run_eval_pipeline.py`.

Ручные скрипты ниже требуют уже сгенерированных predictions:

```bash
python scripts/score_generations.py --mode general --predictions reports/metrics/general_dev_sft.jsonl --baseline reports/metrics/general_dev_base.jsonl
python scripts/score_generations.py --mode autocheck --predictions reports/metrics/plant_dev_sft.jsonl
python scripts/run_judge.py --input reports/metrics/plant_dev_pair.jsonl --limit 20
```

Правило чекпоинта: судья по plant_dev ∪ garden_dev не хуже базы; general_dev drop ≤ 5 п.п.

In [ ]:
from src.eval.checkers import check_no_invented_experience
print(check_no_invented_experience('Water when the top inch is dry; I think a terracotta pot helps.'))
print(check_no_invented_experience("I've grown these for years and in my garden they never rot."))